In [0]:
WITH category_monthly_sales AS (
    SELECT
        d.year,
        d.month,
        d.year_month,
        p.product_category,

        SUM(f.price) AS monthly_category_sales

    FROM `e-commerce`.ecommerce_gold.fact_order_items f

    JOIN `e-commerce`.ecommerce_gold.dim_date d
        ON f.purchase_date_key = d.date_key

    JOIN `e-commerce`.ecommerce_gold.dim_products p
        ON f.product_sk = p.product_sk

    GROUP BY
        d.year,
        d.month,
        d.year_month,
        p.product_category
),

category_growth AS (
    SELECT
        *,

        LAG(monthly_category_sales) OVER (
            PARTITION BY product_category
            ORDER BY year, month
        ) AS previous_month_sales

    FROM category_monthly_sales
)

SELECT
    year_month,
    product_category,

    ROUND(monthly_category_sales, 2) AS monthly_sales,

    ROUND(
        monthly_category_sales - previous_month_sales,
        2
    ) AS sales_change,

    ROUND(
        100.0 * (
            monthly_category_sales - previous_month_sales
        ) / NULLIF(previous_month_sales, 0),
        2
    ) AS category_mom_growth_pct

FROM category_growth

ORDER BY year, month, monthly_sales DESC;